# 🧠 Deep Learning: Convolutional Neural Networks (PyTorch)
Since you hold DataCamp certifications in PyTorch, we will build a custom Convolutional Neural Network (CNN) to classify handwritten digits using the famous MNIST dataset.

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np

# Set device to GPU if available, else CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cpu


## 1. Data Loading & Transformation
We convert the images into PyTorch Tensors and normalize them so pixel values are roughly between -1 and 1.

In [2]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

# Download and load training data
train_dataset = torchvision.datasets.MNIST(root='./data', train=True, transform=transform, download=True)
train_loader = torch.utils.data.DataLoader(dataset=train_dataset, batch_size=64, shuffle=True)

# Download and load testing data
test_dataset = torchvision.datasets.MNIST(root='./data', train=False, transform=transform, download=True)
test_loader = torch.utils.data.DataLoader(dataset=test_dataset, batch_size=64, shuffle=False)

print(f"Training images: {len(train_dataset)}")
print(f"Testing images: {len(test_dataset)}")

100%|██████████| 9.91M/9.91M [00:02<00:00, 4.22MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 353kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 2.89MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 1.80MB/s]

Training images: 60000
Testing images: 10000


## 2. Define the CNN Architecture
We will build a network with two Convolutional layers (to extract edges and loops) followed by Max Pooling, and two Fully Connected (Linear) layers to make the final prediction.

In [3]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        # Conv1: 1 input channel (grayscale), 16 output channels, 3x3 kernel
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        # Conv2: 16 input channels, 32 output channels, 3x3 kernel
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        
        # Fully connected layers
        # Image is 28x28. Pooled twice -> 7x7. 32 channels * 7 * 7 = 1568 features
        self.fc1 = nn.Linear(32 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, 10) # 10 output classes (digits 0-9)
        self.relu = nn.ReLU()
        
    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))
        x = self.pool(self.relu(self.conv2(x)))
        x = x.view(-1, 32 * 7 * 7) # Flatten
        x = self.relu(self.fc1(x))
        x = self.fc2(x) # No softmax here because CrossEntropyLoss applies it internally
        return x

model = SimpleCNN().to(device)
print(model)

SimpleCNN(
  (conv1): Conv2d(1, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (fc1): Linear(in_features=1568, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=10, bias=True)
  (relu): ReLU()
)


## 3. Loss Function & Optimizer
We use Cross Entropy Loss (standard for multi-class classification) and the Adam optimizer to perform gradient descent.

In [4]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

## 4. Training Loop
This is where the magic happens! We will train the model for just 1 Epoch for speed, but you can increase this to get better accuracy.

In [5]:
num_epochs = 1

for epoch in range(num_epochs):
    running_loss = 0.0
    for i, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)
        
        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        # Backward pass and optimize
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        if (i+1) % 300 == 0:
            print(f'Epoch [{epoch+1}/{num_epochs}], Step [{i+1}/{len(train_loader)}], Loss: {running_loss/300:.4f}')
            running_loss = 0.0

print("Training Complete!")

Epoch [1/1], Step [300/938], Loss: 0.4042
Epoch [1/1], Step [600/938], Loss: 0.1042
Epoch [1/1], Step [900/938], Loss: 0.0799
Training Complete!


## 5. Evaluation
Let's see how accurate the model is on data it has never seen before.

In [6]:
model.eval() # Set model to evaluation mode
correct = 0
total = 0
with torch.no_grad(): # Don't calculate gradients to save memory
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f'Accuracy of the network on the 10000 test images: {100 * correct / total:.2f} %')

Accuracy of the network on the 10000 test images: 98.11 %
